# Baseline Model & Forecasting Experiments

## Objective

Establish a seasonal-naive baseline (this week = same week last year) as the minimum bar for any forecasting model. Per the time-series analysis, the series has weak trend and concentrated holiday seasonality, which is exactly what a seasonal-naive baseline captures without any fitting — this determines whether more complex models actually add value.

In [3]:
import pandas as pd
import numpy as np

model_data = pd.read_csv("../data/processed/store_week_features.csv", parse_dates=["Date"])

dates_sorted = sorted(model_data["Date"].unique())
n = len(dates_sorted)
train_end = dates_sorted[int(n * 0.70)]
val_end = dates_sorted[int(n * 0.85)]

train_set = model_data[model_data["Date"] <= train_end].copy()
val_set = model_data[(model_data["Date"] > train_end) & (model_data["Date"] <= val_end)].copy()
test_set = model_data[model_data["Date"] > val_end].copy()

print(len(train_set), len(val_set), len(test_set))

2880 630 585


## Seasonal-Naive Baseline

**Decision:** Predict Weekly_Sales for each store-week using lag_52 (same store, same week, one year earlier) — already computed and leakage-checked in feature engineering. No fitting required; this is a pure lookup.

In [6]:
val_set["baseline_pred"] = val_set["lag_52"]

print(val_set[["Store","Date","Weekly_Sales","baseline_pred"]].head(10))
print()
print("Any missing baseline predictions in val_set:", val_set["baseline_pred"].isnull().sum())

    Store       Date  Weekly_Sales  baseline_pred
64      1 2012-04-27    1468928.37     1455090.69
65      1 2012-05-04    1684519.99     1629391.28
66      1 2012-05-11    1611096.05     1604775.58
67      1 2012-05-18    1595901.87     1428218.27
68      1 2012-05-25    1555444.55     1466046.67
69      1 2012-06-01    1624477.58     1635078.41
70      1 2012-06-08    1697230.96     1588948.32
71      1 2012-06-15    1630607.00     1532114.86
72      1 2012-06-22    1527845.81     1438830.15
73      1 2012-06-29    1540421.49     1488538.09

Any missing baseline predictions in val_set: 0


## Evaluation Metrics

**Decision:** Use MAE, RMSE, and WAPE.

**Why:** MAE gives an average error in the same units as sales (easy to communicate to a business stakeholder). RMSE penalizes large misses more heavily, which matters here since we observed extreme spikes (holiday weeks) that a model could badly miss. WAPE (weighted absolute percentage error = sum of absolute errors / sum of actuals) is used instead of plain MAPE because MAPE is undefined/unstable when actual sales are near zero, and gives a normalized error that's comparable across stores of very different sizes — useful since we found Type A/B/C stores differ by an order of magnitude in scale.

In [9]:
def evaluate(y_true, y_pred, label=""):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))
    print(f"{label} MAE: {mae:,.2f}")
    print(f"{label} RMSE: {rmse:,.2f}")
    print(f"{label} WAPE: {wape:.4f}")
    return {"MAE": mae, "RMSE": rmse, "WAPE": wape}

baseline_metrics = evaluate(val_set["Weekly_Sales"], val_set["baseline_pred"], label="Seasonal-Naive Baseline")

Seasonal-Naive Baseline MAE: 54,807.97
Seasonal-Naive Baseline RMSE: 78,012.51
Seasonal-Naive Baseline WAPE: 0.0523


In [15]:
val_set["abs_error"] = np.abs(val_set["Weekly_Sales"] - val_set["baseline_pred"])

type_error = val_set.groupby("Type").apply(
    lambda d: pd.Series({
        "WAPE": d["abs_error"].sum() / d["Weekly_Sales"].abs().sum(),
        "MAE": d["abs_error"].mean()
    })
)
print(type_error)

          WAPE           MAE
Type                        
A     0.052218  72039.665130
B     0.051156  41893.064874
C     0.058122  28217.287738


/var/folders/l0/_xc3x7m10hxf3hjl2cj6gz2c0000gn/T/ipykernel_7343/1856070898.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  type_error = val_set.groupby("Type").apply(


### Findings: Seasonal-Naive Baseline

**Observed:** The seasonal-naive baseline (predict this week = same week last year) achieves MAE 54,807.97, RMSE 78,012.51, and WAPE 0.0523 on the validation set — meaning predictions are off by about 5.2% of total sales on average, without any model fitting at all. This is a strong baseline, consistent with the concentrated, calendar-recurring seasonality found earlier.

By store Type, WAPE is 0.0522 (A), 0.0512 (B), and 0.0581 (C). This is a genuine surprise: Type C has the worst baseline error despite being the smallest and, per the earlier store-level comparison, the least volatile Type by average coefficient of variation. The likely explanation is sample size — only 6 Type C stores exist, so the group average is more sensitive to one or two individual stores with an atypical year-over-year pattern, rather than Type C being intrinsically harder to forecast as a category.

**Decision:** Any real forecasting model must beat WAPE 0.0523 overall to be worth using over the free seasonal-naive lookup. Store Type C's baseline performance should be checked at the individual-store level (not just the group average) once we get to store-level error analysis, since a 6-store group average can be misleading.

**Why:** Without this baseline, a model with a WAPE of, say, 0.07 might look "reasonable" in isolation but would actually be worse than doing nothing. This number is the actual bar, not an assumed one.